<a href="https://colab.research.google.com/github/Shibin2000/ecommerce-medallion-pipeline/blob/main/ecommerce_medallion_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Libraries

!pip install duckdb plotly pyspark --quiet




error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import pandas as pd
import numpy as np
import duckdb
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

N = 100_000

categories = ['Electronics','Fashion','Home & Garden','Sports',
              'Books','Beauty','Toys','Food & Beverage','Automotive']
cities   = ['Houston','Dallas','Austin','New York','Chicago','Los Angeles']
payments = ['Credit Card','Debit Card','PayPal','Cash']
statuses = ['delivered','shipped','cancelled','returned','pending']
reasons  = ['Defective','Wrong item','Changed mind']

start = datetime(2023, 1, 1)
dates = [start + timedelta(days=random.randint(0, 730)) for _ in range(N)]

df_bronze = pd.DataFrame({
    'order_id':        [f'ORD-{i:07d}' for i in range(N)],
    'customer_id':     [f'CUST-{random.randint(1,20000):06d}' for _ in range(N)],
    'product_id':      [f'PROD-{random.randint(1, 5000):05d}' for _ in range(N)],
    'category':        [random.choice(categories) for _ in range(N)],
    'order_date':      dates,
    'quantity':        np.random.randint(1, 6, N),
    'unit_price':      np.random.uniform(5, 500, N).round(2),
    'discount_pct':    np.random.randint(0, 20, N),
    'shipping_cost':   np.random.uniform(2, 20, N).round(2),
    'customer_rating': np.random.randint(1, 6, N),
    'payment_method':  [random.choice(payments) for _ in range(N)],
    'city':            [random.choice(cities) for _ in range(N)],
    'status':          [random.choice(statuses) for _ in range(N)],
    'return_reason':   [random.choice(reasons + [None]*3) for _ in range(N)] # 75% chance of no return reason
})

# Convert str columns to pd.StringDtype for DuckDB 1.x compatibility
str_cols = ['order_id','customer_id','product_id','category',
            'payment_method','city','status','return_reason']
for col in str_cols:
    df_bronze[col] = df_bronze[col].astype(pd.StringDtype())

# Save df_bronze to DuckDB
with duckdb.connect('ecommerce_lakehouse.db') as conn:
    conn.execute("DROP TABLE IF EXISTS bronze_orders")
    conn.register("df_bronze_view", df_bronze)
    conn.execute("CREATE TABLE bronze_orders AS SELECT * FROM df_bronze_view")
    conn.unregister("df_bronze_view")

Libraries loaded: pandas, numpy, duckdb, datetime


In [3]:
# cleaning up the bronze data here
# removing rows we can't trust and adding columns we'll need later
import duckdb, pandas as pd

# reading fresh from db so this works even after a kernel restart
with duckdb.connect('ecommerce_lakehouse.db') as conn:
    df_bronze = conn.execute("SELECT * FROM bronze_orders").fetchdf()

df_silver = df_bronze.copy()

before    = len(df_silver)
df_silver = df_silver[df_silver['unit_price'] > 0]
df_silver = df_silver[df_silver['quantity']   > 0]
df_silver = df_silver[df_silver['status']    != 'cancelled']  # no point keeping cancelled orders

df_silver['order_date']      = pd.to_datetime(df_silver['order_date'])
df_silver['customer_rating'] = df_silver['customer_rating'].fillna(0).astype(float)
df_silver['return_reason']   = df_silver['return_reason'].fillna('No Return')

# building up the revenue columns step by step
df_silver['gross_amount']    = (df_silver['unit_price'] * df_silver['quantity']).round(2)
df_silver['discount_amount'] = (df_silver['gross_amount'] * df_silver['discount_pct'] / 100).round(2)
df_silver['net_amount']      = (df_silver['gross_amount'] - df_silver['discount_amount']).round(2)
df_silver['total_amount']    = (df_silver['net_amount'] + df_silver['shipping_cost']).round(2)

# these make grouping by time period much easier downstream
df_silver['order_year']     = df_silver['order_date'].dt.year
df_silver['order_month']    = df_silver['order_date'].dt.month
df_silver['order_quarter']  = df_silver['order_date'].dt.quarter
df_silver['order_day_name'] = df_silver['order_date'].dt.day_name().astype(pd.StringDtype())
df_silver['is_weekend']     = df_silver['order_day_name'].isin(['Saturday','Sunday'])
df_silver['is_returned']    = df_silver['status'] == 'returned'

# Convert ALL remaining object/str cols to pd.StringDtype before writing to DuckDB
for col in df_silver.columns:
    if df_silver[col].dtype == object or str(df_silver[col].dtype) == 'str':
        df_silver[col] = df_silver[col].astype(pd.StringDtype())

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    conn.execute("DROP TABLE IF EXISTS silver_orders")
    conn.register("df_silver_view", df_silver)
    conn.execute("CREATE TABLE silver_orders AS SELECT * FROM df_silver_view")
    conn.unregister("df_silver_view")
    silver_count = conn.execute("SELECT COUNT(*) FROM silver_orders").fetchone()[0]

removed = before - silver_count
print(f"SILVER — {silver_count:,} clean rows")
print(f"  Removed {removed:,} bad rows ({removed/before*100:.1f}%)")

SILVER — 80,090 clean rows
  Removed 19,910 bad rows (19.9%)


In [4]:
# building the gold layer — pre-aggregated tables for analytics
# four tables: daily sales, by category, by customer, by city
import duckdb

with duckdb.connect('ecommerce_lakehouse.db') as conn:

    # daily numbers — good for trend analysis
    conn.execute("DROP TABLE IF EXISTS gold_daily_sales")
    conn.execute("""
        CREATE TABLE gold_daily_sales AS
        SELECT
            order_date::DATE               AS sale_date,
            order_year, order_month, order_quarter,
            COUNT(*)                       AS total_orders,
            COUNT(DISTINCT customer_id)    AS unique_customers,
            SUM(quantity)                  AS total_items_sold,
            ROUND(SUM(gross_amount),    2) AS gross_revenue,
            ROUND(SUM(discount_amount), 2) AS total_discounts,
            ROUND(SUM(net_amount),      2) AS net_revenue,
            ROUND(SUM(total_amount),    2) AS total_revenue,
            ROUND(AVG(total_amount),    2) AS avg_order_value,
            SUM(CASE WHEN is_returned THEN 1 ELSE 0 END) AS returns
        FROM silver_orders
        GROUP BY order_date::DATE, order_year, order_month, order_quarter
        ORDER BY sale_date
    """)

    # which categories are actually making money
    conn.execute("DROP TABLE IF EXISTS gold_category_metrics")
    conn.execute("""
        CREATE TABLE gold_category_metrics AS
        SELECT
            category,
            COUNT(*)                       AS total_orders,
            ROUND(SUM(total_amount),    2) AS total_revenue,
            ROUND(AVG(total_amount),    2) AS avg_order_value,
            ROUND(AVG(customer_rating), 2) AS avg_rating,
            ROUND(AVG(discount_pct),    2) AS avg_discount_pct,
            ROUND(SUM(CASE WHEN is_returned THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS return_rate_pct
        FROM silver_orders
        GROUP BY category
        ORDER BY total_revenue DESC
    """)

    # segmenting customers by how much they've spent total
    conn.execute("DROP TABLE IF EXISTS gold_customer_segments")
    conn.execute("""
        CREATE TABLE gold_customer_segments AS
        SELECT
            customer_id,
            COUNT(*)                    AS total_orders,
            ROUND(SUM(total_amount), 2) AS lifetime_value,
            ROUND(AVG(total_amount), 2) AS avg_order_value,
            MIN(order_date::DATE)       AS first_order,
            MAX(order_date::DATE)       AS last_order,
            CASE
                WHEN SUM(total_amount) > 2000 THEN 'VIP'
                WHEN SUM(total_amount) > 1000 THEN 'Premium'
                WHEN SUM(total_amount) >  500 THEN 'Regular'
                ELSE 'New'
            END AS customer_segment
        FROM silver_orders
        GROUP BY customer_id
        ORDER BY lifetime_value DESC
    """)

    # city breakdown
    conn.execute("DROP TABLE IF EXISTS gold_city_metrics")
    conn.execute("""
        CREATE TABLE gold_city_metrics AS
        SELECT
            city,
            COUNT(*)                    AS total_orders,
            ROUND(SUM(total_amount), 2) AS total_revenue,
            ROUND(AVG(total_amount), 2) AS avg_order_value,
            COUNT(DISTINCT customer_id) AS unique_customers
        FROM silver_orders
        GROUP BY city
        ORDER BY total_revenue DESC
    """)

    print("GOLD — Tables created:")
    for t in ['gold_daily_sales','gold_category_metrics','gold_customer_segments','gold_city_metrics']:
        n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  {t:<30} {n:,} rows")

GOLD — Tables created:
  gold_daily_sales               731 rows
  gold_category_metrics          9 rows
  gold_customer_segments         19,584 rows
  gold_city_metrics              6 rows


In [5]:
# simulating a daily incremental load
# instead of reloading everything from scratch we just append new rows
# the trick is checking the max date already loaded and only taking rows after that
import duckdb, pandas as pd, numpy as np, random
from datetime import datetime

np.random.seed(99)
random.seed(99)

BATCH = 300
today = datetime(2026, 3, 19)
categories = ['Electronics','Fashion','Home & Garden','Sports',
              'Books','Beauty','Toys','Food & Beverage','Automotive']

df_new = pd.DataFrame({
    'order_id':        [f'ORD-NEW-{i:04d}' for i in range(BATCH)],
    'customer_id':     [f'CUST-{random.randint(1,20000):06d}' for _ in range(BATCH)],
    'product_id':      [f'PROD-{random.randint(1, 5000):05d}' for _ in range(BATCH)],
    'category':        [random.choice(categories) for _ in range(BATCH)],
    'order_date':      [today] * BATCH,
    'quantity':        np.random.randint(1, 10, BATCH),
    'unit_price':      np.round(np.random.uniform(5, 500, BATCH), 2),
    'discount_pct':    np.random.choice([0, 5, 10, 15, 20, 25], BATCH),
    'shipping_cost':   np.round(np.random.uniform(0, 30, BATCH), 2),
    'customer_rating': np.where(np.random.random(BATCH) > 0.2, np.random.randint(1,6,BATCH), np.nan),
    'payment_method':  np.random.choice(['Credit Card','Debit Card','PayPal','Cash'], BATCH),
    'city':            np.random.choice(['Houston','Dallas','Austin','New York','Chicago','Los Angeles'], BATCH),
    'status':          np.random.choice(['delivered','shipped','pending'], BATCH),
    'return_reason':   None,
})

# Convert str cols for DuckDB 1.x
for _df in [df_new, df_append] if 'df_append' in dir() else [df_new]:
    for col in _df.columns:
        if str(_df[col].dtype) in ('object', 'str'):
            _df[col] = _df[col].astype(__import__('pandas').StringDtype())

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    last = conn.execute("SELECT MAX(order_date) FROM bronze_orders").fetchone()[0]
    df_new['order_date'] = pd.to_datetime(df_new['order_date'])
    df_append = df_new[df_new['order_date'] > pd.Timestamp(last)]

    if len(df_append) > 0:
        # Use INSERT BY NAME to match columns by name, not by position
        conn.execute("INSERT INTO bronze_orders BY NAME SELECT * FROM df_append")
        total = conn.execute("SELECT COUNT(*) FROM bronze_orders").fetchone()[0]
        print(f"Incremental load — {len(df_append):,} new rows appended")
        print(f"Bronze total now  : {total:,}")
    else:
        print("Nothing new to load — already up to date.")

Incremental load — 300 new rows appended
Bronze total now  : 100,300


In [6]:
# PySpark cell — runs in Colab/local with Java 17 installed
# Skipped during automated execution (requires Java runtime)
import sys
if 'google.colab' in sys.modules or False:
    import os, duckdb
    from pyspark.sql import SparkSession
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

    with duckdb.connect('ecommerce_lakehouse.db') as conn:
        df_silver = conn.execute("SELECT * FROM silver_orders").fetchdf()

    spark = SparkSession.builder \
        .appName("ECommerce_Medallion") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    sdf = spark.createDataFrame(df_silver)
    sdf.createOrReplaceTempView("sales")

    print("── Top Categories by Revenue ──")
    spark.sql("""
        SELECT category, COUNT(*) AS orders,
               ROUND(SUM(total_amount), 2) AS revenue,
               ROUND(AVG(customer_rating), 2) AS avg_rating
        FROM sales GROUP BY category ORDER BY revenue DESC
    """).show()
else:
    print("PySpark cell skipped — run in Colab or with Java 17 installed.")
    print("All other pipeline results are shown in the cells above and below.")


PySpark cell skipped — run in Colab or with Java 17 installed.
All other pipeline results are shown in the cells above and below.


In [7]:
# querying the gold layer to get some actual business answers
import duckdb

with duckdb.connect('ecommerce_lakehouse.db') as conn:

    print("── Category Performance ──")
    print(conn.execute("""
        SELECT category, total_revenue, avg_order_value, avg_rating, return_rate_pct
        FROM gold_category_metrics ORDER BY total_revenue DESC
    """).fetchdf().to_string(index=False))

    print("\n── Customer Segments ──")
    print(conn.execute("""
        SELECT customer_segment, COUNT(*) AS customers,
               ROUND(AVG(lifetime_value), 2) AS avg_ltv,
               ROUND(SUM(lifetime_value), 2) AS segment_revenue
        FROM gold_customer_segments
        GROUP BY customer_segment ORDER BY avg_ltv DESC
    """).fetchdf().to_string(index=False))

    print("\n── City Rankings ──")
    print(conn.execute("""
        SELECT city, total_orders, total_revenue, avg_order_value, unique_customers
        FROM gold_city_metrics ORDER BY total_revenue DESC
    """).fetchdf().to_string(index=False))

    # is revenue actually growing quarter over quarter?
    print("\n── Quarter over Quarter ──")
    print(conn.execute("""
        SELECT order_year, order_quarter,
               ROUND(SUM(total_revenue), 2) AS quarterly_revenue,
               SUM(total_orders) AS total_orders
        FROM gold_daily_sales
        GROUP BY order_year, order_quarter
        ORDER BY order_year, order_quarter
    """).fetchdf().to_string(index=False))

── Category Performance ──
       category  total_revenue  avg_order_value  avg_rating  return_rate_pct
        Fashion     6357813.28           704.08        3.03            25.23
     Automotive     6315505.41           703.05        3.01            25.01
          Books     6314092.44           697.84        3.01            25.17
  Home & Garden     6247340.23           705.75        2.99            24.57
         Beauty     6208765.01           696.13        2.99            25.40
    Electronics     6195096.39           697.33        3.02            25.05
         Sports     6174132.01           699.46        2.98            24.48
Food & Beverage     6083998.69           692.62        3.00            24.80
           Toys     6056321.23           691.12        3.01            24.59

── Customer Segments ──
customer_segment  customers  avg_ltv  segment_revenue
             VIP      12536  3808.62      47744823.41
         Premium       4384  1522.10       6672905.61
         Regular

In [8]:
# bar chart — revenue by category, colored by avg rating
# green = high rated, red = low rated
import plotly.express as px, duckdb

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    df = conn.execute("SELECT * FROM gold_category_metrics ORDER BY total_revenue DESC").fetchdf()

fig = px.bar(df, x='category', y='total_revenue', color='avg_rating',
             text='total_revenue', title='Revenue by Product Category',
             labels={'total_revenue':'Revenue ($)','category':'Category'},
             color_continuous_scale='RdYlGn')
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-30, plot_bgcolor='white', yaxis=dict(gridcolor='#eee'))
fig.show()

In [9]:
# rolling up to weekly so the chart isn't too noisy
import plotly.express as px, pandas as pd, duckdb

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    df = conn.execute("SELECT sale_date, total_revenue FROM gold_daily_sales ORDER BY sale_date").fetchdf()

df['sale_date'] = pd.to_datetime(df['sale_date'])
df['week']      = df['sale_date'].dt.to_period('W').astype(str)
df_weekly       = df.groupby('week', as_index=False)['total_revenue'].sum()

fig = px.line(df_weekly, x='week', y='total_revenue',
              title='Weekly Revenue Trend (2023–2025)',
              labels={'total_revenue':'Revenue ($)','week':'Week'}, markers=True)
fig.update_layout(xaxis_tickangle=-45, plot_bgcolor='white', yaxis=dict(gridcolor='#eee'))
fig.show()

In [10]:
# donut chart showing revenue share by customer tier
# usually VIP is a small % of customers but drives a big % of revenue
import plotly.express as px, duckdb

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    df = conn.execute("""
        SELECT customer_segment, COUNT(*) AS customers,
               ROUND(SUM(lifetime_value), 2) AS total_revenue
        FROM gold_customer_segments
        GROUP BY customer_segment ORDER BY total_revenue DESC
    """).fetchdf()

fig = px.pie(df, values='total_revenue', names='customer_segment',
             title='Revenue by Customer Segment', hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdBu)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [11]:
# quick sanity checks across all layers
# if any of these fail something went wrong upstream
import duckdb

passed, failed = 0, 0

def check(label, got, expected):
    global passed, failed
    ok = got == expected
    passed += ok
    failed += (not ok)
    print(f"  {'✓' if ok else '✗'}  {label}: {got}  (expected {expected})")

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    print("SILVER:")
    check("no negative prices",   conn.execute("SELECT COUNT(*) FROM silver_orders WHERE unit_price < 0").fetchone()[0], 0)
    check("no zero quantities",   conn.execute("SELECT COUNT(*) FROM silver_orders WHERE quantity = 0").fetchone()[0], 0)
    check("no null customer ids", conn.execute("SELECT COUNT(*) FROM silver_orders WHERE customer_id IS NULL").fetchone()[0], 0)

    print("\nGOLD:")
    check("9 categories", conn.execute("SELECT COUNT(*) FROM gold_category_metrics").fetchone()[0], 9)
    check("4 segments",   conn.execute("SELECT COUNT(DISTINCT customer_segment) FROM gold_customer_segments").fetchone()[0], 4)

    rev = conn.execute("SELECT SUM(total_revenue) FROM gold_category_metrics").fetchone()[0]
    print(f"\n  total revenue: ${rev:,.2f}")

print(f"\n  {'all checks passed ✓' if failed == 0 else str(failed) + ' failed ✗'}")

SILVER:
  ✓  no negative prices: 0  (expected 0)
  ✓  no zero quantities: 0  (expected 0)
  ✓  no null customer ids: 0  (expected 0)

GOLD:
  ✓  9 categories: 9  (expected 9)
  ✓  4 segments: 4  (expected 4)

  total revenue: $55,953,064.69

  all checks passed ✓


In [12]:
# final summary — row counts for every table we built
import duckdb

print("=" * 55)
print("MEDALLION PIPELINE — DONE")
print("=" * 55)

layers = [
    ("BRONZE", "bronze_orders",          "raw, no transformations"),
    ("SILVER", "silver_orders",          "cleaned and enriched"),
    ("GOLD",   "gold_daily_sales",       "daily KPIs"),
    ("GOLD",   "gold_category_metrics",  "by category"),
    ("GOLD",   "gold_customer_segments", "customer segments"),
    ("GOLD",   "gold_city_metrics",      "by city"),
]

with duckdb.connect('ecommerce_lakehouse.db') as conn:
    for layer, table, desc in layers:
        n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  [{layer}]  {table:<30} {n:>8,} rows  —  {desc}")

MEDALLION PIPELINE — DONE
  [BRONZE]  bronze_orders                   100,300 rows  —  raw, no transformations
  [SILVER]  silver_orders                    80,090 rows  —  cleaned and enriched
  [GOLD]  gold_daily_sales                    731 rows  —  daily KPIs
  [GOLD]  gold_category_metrics                 9 rows  —  by category
  [GOLD]  gold_customer_segments           19,584 rows  —  customer segments
  [GOLD]  gold_city_metrics                     6 rows  —  by city
